In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

# Load the dataset
df = pd.read_csv("C:\\Users\\tejas\\OneDrive\\Desktop\\Textile_python_project\\textile_company_data_10000.csv")

# Basic info
print("Shape:", df.shape)
print("\nColumns:", df.columns.tolist())
print("\nData types:\n", df.dtypes)
print("\nFirst 5 rows:\n", df.head())
print("\nMissing values:\n", df.isnull().sum())


Shape: (10000, 11)

Columns: ['Year', 'Product Name', 'Profit', 'Loss', 'Total Sale', 'Total Manufacturing', 'Remaining Products', 'Total No of Workers', 'Salary of Workers', 'Raw Material Cost', 'Production Cost']

Data types:
 Year                     int64
Product Name            object
Profit                   int64
Loss                   float64
Total Sale               int64
Total Manufacturing      int64
Remaining Products       int64
Total No of Workers      int64
Salary of Workers        int64
Raw Material Cost        int64
Production Cost          int64
dtype: object

First 5 rows:
    Year Product Name  Profit     Loss  Total Sale  Total Manufacturing  \
0  2022      Lehenga       0  28862.0       50301                50301   
1  2023      Lehenga   97331      0.0      194662                97331   
2  2020        Pants   78326      0.0      156652                78326   
3  2022      Dupatta   97670      0.0      195340                97670   
4  2022      Lehenga   30782  

In [2]:
# Clean Loss column (convert to int where possible)
df['Loss'] = df['Loss'].apply(lambda x: int(x) if x == int(x) else 0)

# Calculate net profit: Profit - Loss
df['Net Profit'] = df['Profit'] - df['Loss']

# Profit margin: (Net Profit / Total Sale) * 100
df['Profit Margin %'] = (df['Net Profit'] / df['Total Sale']) * 100

print("Updated sample:\n", df[['Profit', 'Loss', 'Net Profit', 'Profit Margin %']].head())
print("\nNet Profit stats:\n", df['Net Profit'].describe())


Updated sample:
    Profit   Loss  Net Profit  Profit Margin %
0       0  28862      -28862       -57.378581
1   97331      0       97331        50.000000
2   78326      0       78326        50.000000
3   97670      0       97670        50.000000
4   30782      0       30782        50.000000

Net Profit stats:
 count    10000.000000
mean     42347.709000
std      41419.909941
min     -49996.000000
25%      25565.750000
50%      49404.000000
75%      75186.500000
max      99996.000000
Name: Net Profit, dtype: float64


In [3]:
#Net profit averages ~48k per record, with margins varying by product.

## Key Statistics

In [4]:
# Summary stats
print(df.describe())

# Product distribution
print("\nProducts count:\n", df['Product Name'].value_counts())

# Yearly sales avg
print("\nAvg Total Sale by Year:\n", df.groupby('Year')['Total Sale'].mean())


              Year        Profit          Loss     Total Sale  \
count  10000.00000  10000.000000  10000.000000   10000.000000   
mean    2021.49430  48150.293700   5802.584700  110788.196000   
std        1.12138  31489.799839  12851.196275   45731.253153   
min     2020.00000      0.000000      0.000000   40028.000000   
25%     2020.00000  25565.750000      0.000000   71593.500000   
50%     2021.00000  49404.000000      0.000000   99358.500000   
75%     2023.00000  75186.500000      0.000000  150373.000000   
max     2023.00000  99996.000000  49996.000000  199992.000000   

       Total Manufacturing  Remaining Products  Total No of Workers  \
count         10000.000000        10000.000000         10000.000000   
mean          62637.902300           94.305400           199.383900   
std           22623.655361           32.211903            57.160497   
min           20014.000000           40.000000           100.000000   
25%           44059.500000           66.000000           15

## Shirts/Kurtas dominate (~1700 each); 2022 has highest avg sales (~112k).

## Visualizations

## 1. Total Sale by Product (Bar Chart)

In [5]:
import plotly.express as px

sales_data = df.groupby('Product Name')['Total Sale'].mean().reset_index()

fig1 = px.bar(sales_data, x='Product Name', y='Total Sale',
              title='Average Total Sale by Product',
              color='Total Sale', color_continuous_scale='Blues')
fig1.update_traces(texttemplate='₹%{y:,.0f}', textposition='auto', textfont_size=14)
fig1.update_layout(height=500, showlegend=False, xaxis_tickangle=45, font_size=12)
fig1.show()


## 2. Net Profit by Year 

In [6]:
profit_data = df.groupby('Year')['Net Profit'].mean().round(0).astype(int).reset_index()

fig2 = px.bar(profit_data, x='Year', y='Net Profit',
              title='Average Net Profit by Year',
              color='Net Profit', color_continuous_scale='Greens')
fig2.update_traces(texttemplate='₹%{y:,.0f}', textposition='auto', textfont_size=14)
fig2.update_layout(height=500, showlegend=False, font_size=12)
fig2.show()



## 3. Profit Margin boxplot

In [7]:
# Box plot shows distribution clearly (median, quartiles, outliers)
fig3 = px.box(df, x='Product Name', y='Profit Margin %',
              title='Profit Margin Distribution by Product (%)',
              color='Product Name',
              color_discrete_sequence=px.colors.qualitative.Set3)
fig3.update_traces(boxpoints='outliers', jitter=0.3)
fig3.update_layout(
    height=500, font_size=12,
    xaxis_tickangle=45,
    yaxis_title='Profit Margin %'
)
fig3.add_hline(y=df['Profit Margin %'].mean(), line_dash="dash", 
               line_color="red",
               annotation_text=f"Overall Avg: {df['Profit Margin %'].mean():.1f}%")
fig3.show()




## 4. Product vs Production Cost

In [8]:
# Pie chart: Production Cost Distribution by Product
cost_pie = df.groupby('Product Name')['Production Cost'].sum().reset_index()

fig4 = px.pie(cost_pie, values='Production Cost', names='Product Name',
              title='Total Production Cost Share by Product (₹)',
              color_discrete_sequence=px.colors.qualitative.Set3)
fig4.update_traces(texttemplate='%{label}<br>₹%{value:,.0f}<br>%{percent}',
                   textposition='inside', textfont_size=12)
fig4.update_layout(height=500, font_size=12, legend=dict(x=1, y=0.5))
fig4.show()


## 5. Top 10 Profitable Records (Bar + Table)

In [9]:
top_profit = df.nlargest(10, 'Net Profit')[['Product Name', 'Net Profit', 'Total Sale', 'Year']]

fig5 = px.bar(top_profit, x='Net Profit', y='Product Name', orientation='h',
              title='Top 10 Most Profitable Records',
              text='Net Profit', color='Total Sale')
fig5.update_traces(texttemplate='₹%{x:,.0f}', textposition='auto')
fig5.update_layout(height=500, font_size=12, xaxis_title='Net Profit (₹)')
fig5.show()

# Bonus: Data table
print(top_profit.to_string(index=False))


Product Name  Net Profit  Total Sale  Year
     Lehenga       99996      199992  2020
     Lehenga       99973      199946  2022
     Lehenga       99951      199902  2023
     Lehenga       99941      199882  2022
       Saree       99922      199844  2022
       Pants       99921      199842  2020
       Pants       99921      199842  2022
       Pants       99919      199838  2022
       Shirt       99908      199816  2021
     Lehenga       99903      199806  2020


## 6. Sales vs Raw Material Cost (Scatter Matrix)

In [10]:
#  used simple scatter
fig6 = px.scatter(df.sample(2000), x='Raw Material Cost', y='Total Sale',
                  size='Production Cost', color='Profit Margin %',
                  hover_name='Product Name',
                  title='Raw Material vs Sales (Size=Prod Cost, Color=Margin)',
                  size_max=15, opacity=0.6)  

fig6.update_layout(height=500, font_size=12)
fig6.show()



## 7. Product Performance Heatmap

In [11]:
heatmap_data = df.groupby(['Product Name', 'Year']).agg({
    'Net Profit': 'mean',
    'Total Sale': 'mean'
}).round(0).reset_index()

fig7 = px.density_heatmap(heatmap_data, x='Year', y='Product Name',
                          z='Net Profit',
                          title='Avg Net Profit Heatmap (₹)',
                          color_continuous_scale='RdYlGn')
fig7.update_layout(height=500, font_size=12)
fig7.show()


## 8. Workers Efficiency (Profit per Worker)

In [12]:
df['Profit per Worker'] = df['Net Profit'] / df['Total No of Workers']

fig8 = px.box(df, x='Product Name', y='Profit per Worker',
              title='Profit Efficiency (₹ per Worker)',
              color='Product Name')
fig8.update_layout(height=500, xaxis_tickangle=45, font_size=12)
fig8.show()


## 9. Cost Breakdown Pie (Avg Across All)

In [13]:
cost_breakdown = df[['Raw Material Cost', 'Production Cost', 'Salary of Workers']].mean().reset_index()
cost_breakdown.columns = ['Cost Type', 'Amount']

fig9 = px.pie(cost_breakdown, values='Amount', names='Cost Type',
              title='Average Cost Breakdown',
              color_discrete_sequence=['gold', 'lightcoral', 'lightblue'])
fig9.update_traces(textinfo='label+percent+value', textfont_size=12)
fig9.update_layout(height=500)
fig9.show()


## 10. Year-over-Year Growth (Line Chart)

In [15]:
import plotly.graph_objects as go
growth = df.groupby('Year').agg({
    'Total Sale': ['mean', 'sum'],
    'Net Profit': ['mean', 'sum']
}).round(0)


years = growth.index.astype(int).tolist()

fig10 = go.Figure()
fig10.add_trace(go.Scatter(x=years, y=growth['Total Sale']['mean'],
                          mode='lines+markers', name='Avg Sale',
                          line=dict(color='blue', width=3),
                          marker=dict(size=10)))
fig10.add_trace(go.Scatter(x=years, y=growth['Net Profit']['mean'],
                          mode='lines+markers', name='Avg Profit',
                          line=dict(color='green', width=3),
                          marker=dict(size=10)))
fig10.update_layout(
    title='YoY Growth Trends (Avg Sale & Profit)',
    height=500, font_size=12,
    xaxis=dict(title='Year', tickmode='array', tickvals=years)
)
fig10.show()

